# 2장 2강: 임베딩 기반 문서 검색 — 실습문제


## 실습 목표

- 검색 대상 본문, 문서 ID, 메타데이터를 구분하고 원본과의 연결을 유지할 수 있다.
- 문서 임베딩과 질문 임베딩의 역할·생성 시점·갱신 조건을 설명할 수 있다.
- 제공된 검색 점수로 Top-K 후보를 선택하고 원문과 관련 정보를 확인할 수 있다.
- 주제 관련성·조건 충족·정보 충분성을 기준으로 검색 결과를 검증할 수 있다.
- 검색 실패 시 확인할 사항과 RAG에서 검색 결과를 활용하는 흐름을 설명할 수 있다.

## 실습 환경 / 데이터

- Python 3, Jupyter Notebook 또는 VS Code 노트북, pandas
- `reviews.csv`, `review_labels.csv`, `aspect_labels.csv`를 노트북과 같은 폴더에 둡니다. 다른 위치라면 준비 코드의 `DATA_DIR`를 수정합니다.
- 앞 강의의 노트북 실행 결과는 필요하지 않습니다. 이 노트북을 위에서 아래로 실행합니다.

| 파일 | 한 행의 의미 | 이번 실습의 역할 |
|---|---|---|
| reviews.csv | 리뷰 한 개 | 검색 대상 본문과 리뷰 ID·메타데이터 확인 |
| review_labels.csv | 리뷰 한 개의 기존 전체 라벨 | 검색된 리뷰의 기존 전체 코드를 참고 정보로 조회 |
| aspect_labels.csv | 리뷰의 속성별 주석 한 개 | 속성명과 evidence_text를 읽어 세부 조건 확인 |

사용자가 지칭한 `review_label`, `aspect_label`의 실제 제공 파일명은 각각 `review_labels.csv`, `aspect_labels.csv`입니다. 세 파일은 `review_id`로 연결합니다. 같은 리뷰에 속성별 주석이 여러 개 있을 수 있으므로, 속성별 행 수를 문서 수로 해석하지 않습니다.

### 교안의 적용 범위

교안의 명언 검색 구조를 제공된 **리뷰 검색**에 적용합니다. `text` 대신 `review_text`, 문서 ID 대신 `review_id`, 작성자·출처 등의 정보 대신 `source`, `domain`, `category`, `product_name`을 사용합니다. 제공된 파일에는 `source_url`이 없으므로 URL을 만들지 않습니다.

이번 강의는 임베딩 알고리즘 구현보다 **문서 구성 → 검색 후보 선택 → 원문 해석 → 적합성 검증**에 집중합니다.

- 리뷰·ID·메타데이터·라벨·근거 표현은 CSV의 실제 데이터입니다.
- 실습의 검색 점수는 교안의 검색 결과 표처럼 **학습용으로 정한 가상 값**입니다. 실제 임베딩 모델로 계산한 점수가 아닙니다.
- `vector_row`는 벡터를 저장한다고 가정했을 때의 행 위치를 나타냅니다. 벡터 자체나 의미 특성값이 아닙니다.
- 실제 임베딩 생성, API 호출, 벡터 데이터베이스 구축, RAG 구현은 요구하지 않습니다. 모델의 검색 성능을 측정한 결과로도 해석하지 않습니다.
- pandas는 파일 읽기·표 구성·정렬을 위한 보조 도구입니다. 사용 구문을 제공하므로 검색 구조와 해석에 집중하세요.
- 기존 `sentiment_code`는 검색 적합성의 정답이나 유사도 점수가 아닙니다. 코드만으로 문서의 조건 충족 여부를 결정하지 않습니다.


## 시작 전에 읽는 개념 정리

### 1. 이번 실습에서 무엇을 배우나요?

사용자가 원하는 것은 점수가 높은 숫자 자체가 아니라 **질문의 조건에 맞는 리뷰의 내용**입니다. 따라서 검색 결과를 받았을 때 어느 리뷰인지 찾아가고, 원문을 읽어 필요한 근거가 있는지 판단할 수 있어야 합니다.

이 노트북에서는 리뷰 4개에 주어진 가상 점수를 붙이고 정렬합니다. 모델이 텍스트를 벡터로 바꾸거나 유사도를 계산하는 부분은 실행하지 않습니다. 앞 강의의 코사인 유사도 계산을 반복하기보다 **이미 주어진 검색 결과를 어떻게 확인하고 활용하는지** 익힙니다.

| 용어 | 쉬운 뜻 | 이번 실습에서 확인할 것 |
|---|---|---|
| 문서 | 검색 대상으로 삼는 텍스트 한 개 | 리뷰 한 행을 한 문서로 사용 |
| 본문 | 문서에 실제로 적힌 내용 | 원문은 `review_text`, 앞뒤 공백을 정리한 본문은 `search_text` |
| 문서 ID | 어느 문서인지 구별하는 식별자 | `review_id`로 여러 표의 같은 리뷰를 연결 |
| 메타데이터 | 본문에 딸린 관련 정보 | 출처·분야·분류·상품명 |
| 임베딩 | 텍스트의 의미적 특징을 숫자 벡터로 표현한 것 | 이 강의에서는 역할만 이해하고 생성하지 않음 |
| 문서 임베딩 | 검색 대상 문서를 표현한 벡터 | 문서를 준비할 때 만들어 저장하는 개념 |
| 질문 임베딩 | 찾으려는 내용을 표현한 벡터 | 새 질문이 들어올 때 생성하는 개념 |
| 검색 순위 | 점수가 높은 순서 | `sort_values()`로 점수의 내림차순 정렬 |
| Top-K | 순위가 높은 문서 K개 | K=2라면 상위 두 문서 선택 |
| 관련성·적합성 | 사용자의 질문에 실제로 맞는 정도 | 원문이 질문의 조건을 뒷받침하는지 판단 |
| 기대 문서 | 질문에 맞는다고 원문을 보고 미리 정한 문서 | 검색 결과에 포함되는지 비교할 기준 |
| RAG | 검색한 문서를 LLM에 제공하여 답변에 활용하는 방식 | 흐름과 검증할 내용만 설명 |

`review_id`가 크다고 더 좋은 리뷰는 아닙니다. ID는 식별자이지 유사도나 평점이 아닙니다. `source`의 SNS는 출처의 종류이며, 특정 게시물의 주소를 나타내지는 않습니다.

### 2. 비슷해 보이는 세 가지 판단을 구분하기

| 판단 기준 | 스스로 해 볼 질문 | 확인 방법 |
|---|---|---|
| 주제 관련성 | 무엇에 관한 글인가요? | 질문과 문서가 다루는 대상·주제를 비교 |
| 조건 충족 | 사용자가 요구한 구체적인 상황인가요? | 색상·착용 조건·불편 경험 등의 직접 표현 확인 |
| 정보 충분성 | 이 내용만으로 요청에 답할 수 있나요? | 필요한 사실이나 사례를 실제로 확인할 수 있는지 검토 |

예를 들어 패딩이라는 주제가 같아도, 사용자가 원하는 것이 ‘두꺼운 옷을 안에 입었을 때 불편했던 사례’라면 그 상황을 직접 말하는지 확인해야 합니다. 조건을 언급하지 않은 문서는 그 조건을 확인하기에 부족합니다. 언급이 없다는 사실만으로 그 경험이 절대 없었다고 결론 내리지는 않습니다.

정보가 충분한지는 **질문에 따라 달라집니다**. 한 사람의 착용 후기는 그 사람이 경험한 사례를 찾는 데 사용할 수 있지만, 상품의 공식 환불 기간을 확인하는 근거로는 부족할 수 있습니다.

### 3. 이 노트북을 읽고 실행하는 순서

| 단계 | 사용하는 정보 | 수행하는 일 |
|---|---|---|
| 실습 준비 | CSV 3개 | 본문·ID·결측치·주석을 확인하고 후보 4개 고정 |
| 필수 1 | 후보와 가상 저장 위치 | 문서 목록을 만들고 위치에서 원문 ID를 찾아 조회 |
| 필수 2 | 첫 질문과 가상 점수 | ID로 연결하고 Top-K를 선택한 뒤 원문으로 판단 |
| 과제 1 | 새 질문과 별도 가상 점수 | 기대 문서를 먼저 정하고 검색 결과와 비교 |
| 실습 마무리 | 코드 출력과 원문 근거 | 검색 결과를 활용할 수 있는 이유와 한계를 설명 |

코드 셀은 위에서 아래로 실행합니다. 예를 들어 필수 2의 `documents`는 필수 1에서 만들고, 과제의 `ranked_results`는 필수 2에서 만듭니다. 중간 셀부터 실행하여 변수가 없다는 오류가 나면 필요한 앞 셀을 먼저 실행하세요.


## 실습 준비

다음 **제공 코드**를 그대로 실행하세요. 파일 읽기와 조회 함수 작성 자체는 평가하지 않습니다.

1. 세 파일의 행·열 수, 컬럼명, 자료형, 결측치를 확인합니다.
2. `reviews.csv` 원본 순서의 앞 4개 리뷰를 이번 검색 후보로 사용합니다.
3. 원문을 보존하고 앞뒤 공백만 정리하여 `search_text`를 만듭니다.
4. 본문 누락·빈 문자열·ID 중복을 확인하고, 글자 수와 출처·분야 분포를 살펴봅니다.
5. 후보의 전체 라벨과 속성별 주석을 ID로 조회합니다.

후보 4개는 소규모 연습용입니다. 전체 데이터셋의 대표 표본이 아닙니다. 아래의 준비 코드는 누락이나 중복이 발견되면 값을 출력하므로 먼저 확인하세요. 제공된 데이터의 후보 4개에는 본문 누락·빈 문자열·ID 중복이 없습니다.


In [1]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path(".")  # 데이터가 다른 폴더에 있으면 수정하세요.
reviews = pd.read_csv(DATA_DIR / "reviews.csv")
review_labels = pd.read_csv(DATA_DIR / "review_labels.csv")
aspect_labels = pd.read_csv(DATA_DIR / "aspect_labels.csv")

for name, table in [("reviews", reviews), ("review_labels", review_labels),
                    ("aspect_labels", aspect_labels)]:
    print(f"\n[{name}] {table.shape[0]:,}행 × {table.shape[1]}열")
    print(pd.DataFrame({"자료형": table.dtypes.astype(str),
                        "결측치": table.isna().sum()}).to_string())

candidates = reviews.head(4).copy().reset_index(drop=True)
candidates["search_text"] = candidates["review_text"].str.strip()
print("\n후보 본문 결측 수:", candidates["search_text"].isna().sum())
print("후보 빈 본문 수:", candidates["search_text"].eq("").sum())
print("후보 ID 중복 수:", candidates["review_id"].duplicated().sum())
print(candidates[["review_id", "source", "domain", "category", "product_name"]].to_string(index=False))
print("\n후보 본문 글자 수:", candidates["search_text"].str.len().tolist())
print(candidates["search_text"].str.len().agg(["min", "mean", "max"]).to_string())
print("\n출처 분포:")
print(candidates["source"].value_counts(dropna=False).to_string())
print("\n분야 분포:")
print(candidates["domain"].value_counts(dropna=False).to_string())

candidate_ids = candidates["review_id"].tolist()
candidate_review_labels = review_labels[review_labels["review_id"].isin(candidate_ids)].copy()
candidate_aspect_labels = aspect_labels[aspect_labels["review_id"].isin(candidate_ids)].copy()
print("\n연결된 전체 라벨 행 수:", len(candidate_review_labels))
print("연결된 속성별 주석 행 수:", len(candidate_aspect_labels))

def show_review_context(review_id):
    row = candidates[candidates["review_id"] == review_id].iloc[0]
    print("\n리뷰 ID:", review_id)
    print("메타데이터:", row[["source", "domain", "category", "product_name"]].to_dict())
    print("원문:", row["review_text"])
    print("기존 전체 라벨:")
    print(candidate_review_labels[candidate_review_labels["review_id"] == review_id].to_string(index=False))
    print("속성별 주석:")
    print(candidate_aspect_labels[candidate_aspect_labels["review_id"] == review_id]
          .sort_values("aspect_no").to_string(index=False))



[reviews] 99,992행 × 6열
                자료형  결측치
review_id     int64    0
source          str    0
domain          str    0
category        str    0
product_name    str   16
review_text     str    0

[review_labels] 99,992행 × 2열
                    자료형   결측치
review_id         int64     0
sentiment_code  float64  1143

[aspect_labels] 373,167행 × 5열
                  자료형  결측치
review_id       int64    0
aspect_no       int64    0
aspect            str    0
evidence_text     str    0
sentiment_code  int64    0

후보 본문 결측 수: 0
후보 빈 본문 수: 0
후보 ID 중복 수: 0
 review_id source domain category  product_name
   1024338    SNS     패션     여성의류   OO 경량 다운 자켓
   1024477    SNS     패션     여성의류 OO 아** 구스코트  
   1025044    SNS     패션     여성의류 OO 아** 구스코트  
   1025046    SNS     패션     여성의류 OO 아** 구스코트  

후보 본문 글자 수: [512, 464, 314, 307]
min     307.00
mean    399.25
max     512.00

출처 분포:
source
SNS    4

분야 분포:
domain
패션    4

연결된 전체 라벨 행 수: 4
연결된 속성별 주석 행 수: 27


### 준비 코드 읽기 — 표와 원문을 먼저 확인하기

`DataFrame`은 행과 열로 이루어진 표입니다. 이번에는 리뷰 본문 표, 전체 라벨 표, 속성별 주석 표를 각각 따로 읽습니다. 세 표의 줄 번호가 같다는 이유로 같은 리뷰라고 생각하지 말고 `review_id`가 같은지 확인하세요.

| 코드·표현 | 하는 일 | 결과를 읽는 방법 |
|---|---|---|
| `Path(".")` | 현재 작업 폴더를 경로로 지정 | CSV가 다른 폴더라면 이 경로를 실제 데이터 폴더로 수정 |
| `DATA_DIR / "reviews.csv"` | 폴더 경로와 파일 이름을 연결 | 여기서 `/`는 나눗셈이 아니라 경로 연결 |
| `pd.read_csv(...)` | CSV를 읽어 표로 만듦 | 파일을 읽는 것만으로 임베딩이 생성되지는 않음 |
| `table.shape[0]`, `table.shape[1]` | 행 수와 열 수를 확인 | 앞의 0은 행, 1은 열을 가리킴 |
| `table.dtypes.astype(str)` | 각 열의 자료형을 문자열로 표시 | 자료형 이름을 보기 위한 것이며 원본 열의 값을 모두 문자로 바꾸는 작업은 아님 |
| `table.isna().sum()` | 열별 결측치 수를 셈 | 값이 비어 있는지 확인하며 숫자 0을 결측치로 취급하지 않음 |
| `reviews.head(4)` | 원본 순서의 앞 네 행 선택 | 무작위 표본이나 유사도가 높은 네 행이 아님 |
| `.copy()` | 선택한 표를 별도로 복사 | 이후 후보 표에 필요한 열을 추가하기 위함 |
| `.reset_index(drop=True)` | 표의 인덱스를 0부터 다시 붙임 | `review_id` 값은 바꾸지 않으며 기존 인덱스 열도 추가하지 않음 |
| `.str.strip()` | 문자열 양쪽 끝의 공백 제거 | 문장 안의 공백·단어·의미를 정제하는 작업 전체를 수행하지는 않음 |
| `.eq("").sum()` | 빈 문자열과 같은 값의 수를 셈 | 결측치 검사와 별도로 내용 없는 문자열을 확인 |
| `.duplicated().sum()` | 앞서 나온 ID와 중복되는 행 수 확인 | 첫 등장은 중복으로 세지 않음 |
| `.str.len()` | 본문의 글자 수 계산 | 단어 개수·벡터 차원·유사도와 구분 |
| `.agg(["min", "mean", "max"])` | 최솟값·평균·최댓값을 한 번에 계산 | 본문 길이의 분포를 간단히 확인 |
| `.value_counts(dropna=False)` | 값별 등장 횟수 계산 | 결측치가 있으면 그것도 분포에 포함 |
| `.tolist()` | 표의 한 열을 파이썬 목록으로 변환 | ID 네 개를 다른 표에서 조회할 준비 |
| `.isin(candidate_ids)` | 후보 ID 목록에 속하는지 행마다 확인 | `True`인 행만 선택하여 후보와 연결된 라벨을 가져옴 |

결측치 수나 중복 수를 출력한다고 자동으로 삭제되는 것은 아닙니다. 이 준비 코드는 문제를 **확인**하는 코드이며, 제공된 후보에는 본문 결측·빈 문자열·ID 중복이 없어서 그대로 진행합니다.

`print()`는 값을 화면에 보여 줍니다. `.to_string(index=False)`는 표를 문자열로 표시하면서 왼쪽 인덱스를 숨깁니다. 표의 행이나 ID 자체를 삭제하는 옵션이 아닙니다. `f"...{값}..."`은 문자열 속에 값을 넣는 표기이고, `\n`은 출력할 때 줄을 바꿉니다.


---
## 필수 1. 검색 대상 문서와 원본 정보 연결

### 문제 1-1. 리뷰 검색용 문서 목록 구성하기

#### 문제 설명

후보 리뷰 4개를 검색할 문서로 준비합니다. 실제로 벡터를 생성하지는 않지만, 문서별 벡터를 저장한다고 가정했을 때 **어떤 벡터 행이 어떤 리뷰에 해당하는지** 알 수 있도록 문서 목록을 구성합니다.

| 가상 벡터 저장 위치 vector_row | review_id |
|---:|---:|
| 0 | 1024338 |
| 1 | 1024477 |
| 2 | 1025044 |
| 3 | 1025046 |

#### 요구사항

1. `candidates`에서 `review_id`, `search_text`, `source`, `domain`, `category`, `product_name`을 선택하여 `documents`를 만드세요. 원래 후보 순서를 유지합니다.
2. `documents["vector_row"] = range(len(documents))`로 가상 벡터 저장 위치를 추가하세요. 실제 임베딩을 생성하는 코드가 아닙니다.
3. `vector_row`, `review_id`, `source`, `domain`, `category`, `product_name`을 순서대로 출력하고, 본문은 각 행의 `search_text`를 별도로 출력하세요.
4. 가상 검색 결과가 `vector_row = 1`을 반환했다고 가정합니다. `documents[documents["vector_row"] == 1]`로 해당 문서를 찾고, 리뷰 ID를 얻어 `show_review_context()`를 실행하세요.

**사용 구문:** `표[["컬럼1", "컬럼2"]].copy()`, `range(len(표))`, `표[조건]`, `표["컬럼"].iloc[0]`, `표.to_string(index=False)`

#### 해석 질문

**Q1.** 이 실습에서 검색할 본문, 문서 ID, 메타데이터는 각각 무엇인가요? `vector_row = 1`에 연결된 리뷰 ID와 상품명은 무엇인가요?

**Q2.** `vector_row`와 `review_id`의 역할은 어떻게 다른가요? 문서 목록만 정렬하고 벡터와의 대응 관계를 갱신하지 않으면 어떤 문제가 생기나요?

**Q3.** 문서 임베딩과 질문 임베딩의 대상·생성 시점·역할을 구분하세요. 새 질문이 들어올 때 모든 문서 임베딩을 다시 만들어야 하나요? 질문·문서 벡터를 비교하려면 어떤 조건이 필요한가요?

**Q4.** 리뷰 본문이 수정되면 무엇을 갱신해야 하나요? 한국어 질문으로 영어 문서를 검색하는 상황이라면 모델의 어떤 특성을 확인해야 하나요? 이 실습에서 실제 임베딩을 생성했나요?

#### 제출 결과

- 문서 목록과 본문을 출력한 코드
- `vector_row = 1`에 연결된 리뷰의 원문·라벨·근거 출력
- Q1~Q4 답변


### 풀이 전 안내 — 문서 목록부터 차근차근

먼저 `documents`의 한 행이 리뷰 한 개를 나타내도록 구성하세요. 열을 고르는 단계와 가상 저장 위치를 붙이는 단계를 구분하면 됩니다. `range(len(documents))`는 0부터 시작하는 위치 번호를 붙이며, 문장 의미를 계산하지 않습니다.

위치를 조회할 때는 `표[표["vector_row"] == 찾을_위치]`로 행을 고른 다음, 그 행의 `review_id` 값을 꺼냅니다. ‘위치로 행 찾기 → ID 확인 → ID로 원문 조회’ 순서를 먼저 말로 설명하고 코드를 작성해 보세요. 결과표의 줄 번호와 문서 ID를 혼동하지 않는 것이 중요합니다.

해석 답변에는 본문·메타데이터·ID가 각각 무엇인지 적고, 문서와 질문 임베딩의 생성 시점도 구분해 보세요. 실제 임베딩 생성 코드를 추가할 필요는 없습니다.


In [3]:
# 필수 1 코드를 작성하세요.
documents = candidates[['review_id','search_text','source','domain','category','product_name']].copy()

documents['vector_row']=range(len(documents))

print(documents[['vector_row','review_id','source','domain','category','product_name']].to_string(index=False))
for vr, text in zip(documents['vector_row'], documents['search_text']):
    print(f'\n[vector_row {vr}] {text}')

hit = documents[documents['vector_row']==1]
hit_review_id=hit['review_id'].iloc[0]
show_review_context(hit_review_id)

 vector_row  review_id source domain category  product_name
          0    1024338    SNS     패션     여성의류   OO 경량 다운 자켓
          1    1024477    SNS     패션     여성의류 OO 아** 구스코트  
          2    1025044    SNS     패션     여성의류 OO 아** 구스코트  
          3    1025046    SNS     패션     여성의류 OO 아** 구스코트  

[vector_row 0] 안녕하세요 이웃님들 반갑습니다. 요즘 날씨가 많이 쌀쌀해졌죠? 요즘 계절에 입으면 딱 좋을 인생 경량 패딩 하나 소개해 드릴게요.  다 함께 go go go~~  제가 입어보고 좋아서 엄마께도 구매해드렸네요. 딱 기본 스타일인데 또 입은 거 보면 깔끔하게 저렴해보이지 않는 디자인이라서 어디서 샀냐고 많이들 물어보는 옷입니다.  엄마도 제가 입은 것 보시더니 탐내셔서  주문해드렸어요. 영하로 내려가는 날씨에는 이것만 입기엔 얇지만 초겨울까지는 운동 갈때 안에 얇은 기능성 반팔 입고 요것만 입어도 꽤 따뜻해요. 색상도 디자인도 무난해서 사무실에 두고 입기도 좋고, 평소에 가볍게 나갈때 코디하기도 좋아요. 앞으로 코트를 입거나 할때 안에 내피로 활용하기도 딱일 듯요! 맘같아선 색깔별로 쟁이고 싶습니다.  20대와 50대 모두 아우르는 디자인이 무난하고 깔끔하면서 고급스러운 옷 추천합니다~~~  설명 끝! 좀 도움이 되셨나요? 그러면 좋아요 꾹 눌러 주시고 전 이만 총총총~~ 항상 여러 이웃님들께 감사드립니다.

[vector_row 1] 드디어 겨울이 찾아왔네요. 이제부터 슬슬 겨울 패딩 장만하셔야지요? 패딩 소개해 드릴게요.   작년부터 눈여겨 보던 OO입니다. 저는 블랙90 사이즈 구매했는데 딱 잘 맞네요. 작년에 비해 저렴하게 구매해서 아주 만족해요. 신제품은 경량패딩으로 되서 3in1기능이더라구요. 경량

### 문서, 질문 임베딩 역할
|상황|실제 임베딩 검색 시스템에서 할 일| 이유|
|---|---|---|
|문서 네 개를 처음 준비|각 문서의 임베딩을 만들고 ID값과 연결|검색 대상을 미리 표현하기 위해|
|본문은 그대로이고 질문만 새로 들어온 경우| 새 질문을 임베딩하고 저장된 문서 임베딩과 비교|찾으려는 내용이 바귀었기 때문|
|특정 리뷰의 본문이 수정됨|해당 문서 임베딩과 원본 연결을 갱신| 이전 의미 표현과 현재 본문이 달라질 수 있기 때문|
| 한국어 질문으로 영어 문서를 검색| 두 언어의 의미를 비교할 수 있는 모델 확인| 언어가 다르더라도 비교할 수 있는 표현이 필요하기 때문|



### 필수 1 답변 작성란

**Q1.** 이 실습에서 검색할 본문, 문서 ID, 메타데이터는 각각 무엇인가요? `vector_row = 1`에 연결된 리뷰 ID와 상품명은 무엇인가요?
- 본문: 검색할 본문은 review_text에서 앞뒤 공백을 정리한 search_text이다.
- 문서 ID는 review_id
- 메타데이터는 source, domain, category, product_name입니다.
- 위치 1에 연결되는 리뷰 ID는 1024477, 상품명는 OO 아** 구스코트, 상품명의 별표는 데이터에 들어가 있는 익명화 표기

**Q2.** `vector_row`와 `review_id`의 역할은 어떻게 다른가요? 문서 목록만 정렬하고 벡터와의 대응 관계를 갱신하지 않으면 어떤 문제가 생기나요?
- vector_row는 가정한 벡터 저장 배열의 행 위치이고, review_id는 원본 리뷰를 식별하는 값입니다.
- 목록만 정리한 뒤 새 행 순서를 이전 벡터 순서로 착각하면 다른 리뷰의 벡터와 본문을 연결할 수 있습니다.
- 정렬이나 데이터 변경 후에도 저장 위치, 문서 ID, 본문의 대응을 유지해야 한다.
- 이번처럼 위치를 명시한 경우에는 현재 화면의 줄 번호가 아니라 저장해둔 대응 값을 사용한다.

**Q3.** 문서 임베딩과 질문 임베딩의 대상·생성 시점·역할을 구분하세요. 새 질문이 들어올 때 모든 문서 임베딩을 다시 만들어야 하나요? 질문·문서 벡터를 비교하려면 어떤 조건이 필요한가요?
|구분|문서 임베딩|질문 임베딩|
|---|---|---|
|대상|검색할 리뷰 본문|사용자가 입력한 질문|
|생성 시점|문서를 검색 대상으로 준비할 때|새로운 검색 질문이 들어올 때|
|역할|검색 대상의 의미적 특징 표현|찾으려는 내용의 의미적 특징 표현|

**Q4.** 리뷰 본문이 수정되면 무엇을 갱신해야 하나요? 한국어 질문으로 영어 문서를 검색하는 상황이라면 모델의 어떤 특성을 확인해야 하나요? 이 실습에서 실제 임베딩을 생성했나요?
- 본문이 바뀌면 해당 문서 임베딩을 다시 생성하고 원본 연결을 갱신해야 합니다.
- 이전 임베딩을 그대로 두면 검색에 쓰는 표현과 현재 본문이 달라질 수 있다.
- 한국어 질문과 영어 문서를 비굫려면 두 언어의 의미를 비교할 수 있는 다국어 검색 지원 여부를 확인합니다.

---
## 필수 2. 검색 순위와 질문 조건 구분

### 문제 2-1. Top-K 후보를 원문으로 검증하기

#### 문제 설명

사용자 질문은 다음과 같습니다.

> **안에 두껍게 입기 불편했다고 직접 말한 패딩 후기를 찾아줘.**

검색 결과가 아래처럼 나왔다고 **가정**합니다. 모든 점수는 학습용으로 만든 값이며 실제 모델 출력이 아닙니다. 질문은 패딩이라는 주제뿐 아니라 **두꺼운 옷을 안에 입었을 때의 불편을 직접 언급했는지**를 요구합니다.

| review_id | 가상_유사도 |
|---:|---:|
| 1024338 | 0.81 |
| 1024477 | 0.85 |
| 1025044 | 0.91 |
| 1025046 | 0.30 |

#### 요구사항

1. 위 표를 `pd.DataFrame()`으로 만들어 `score_table`에 저장하세요.
2. 제공 구문 `score_table.merge(documents, on="review_id", how="left")`로 점수와 원본 정보를 연결하여 `search_results`를 만드세요. `merge()`는 같은 ID의 정보를 옆에 붙이는 보조 구문입니다.
3. `sort_values("가상_유사도", ascending=False)`로 높은 점수부터 정렬하여 `ranked_results`에 저장하세요. Top-1과 Top-2의 ID·점수·상품명·출처를 출력하세요. 상위 K개는 `head(K)`로 선택합니다.
4. Top-2 각각에 대해 `show_review_context()`를 실행하여 실제 원문·라벨·속성별 근거를 확인하세요.
5. Top-2 각각의 `review_id`, `주제 관련성`, `조건 충족`, `정보 충분성`, `판단 근거`를 담은 `relevance_check` 표를 만드세요. 판단 문구는 원문을 읽고 직접 작성합니다. 질문의 조건을 직접 뒷받침할 수 있는지 평가하세요.

#### 해석 질문

**Q1.** Top-1과 Top-2의 리뷰 ID·점수는 무엇인가요? 0.91을 “정답일 확률 91%”로 해석할 수 있나요? 모든 검색 시스템에 같은 점수 기준을 적용해도 되나요?

**Q2.** Top-2 중 질문에 직접 적합한 리뷰는 무엇인가요? 두 리뷰의 실제 표현을 근거로 주제 관련성·조건 충족·정보 충분성을 설명하세요. “사이즈가 딱 맞는다”만으로 두꺼운 옷을 입기 불편하다고 확정할 수 있나요?

**Q3.** 이 가상 결과에서 K=1과 K=2의 차이는 무엇인가요? K를 늘리면 생기는 장단점과 K를 정할 기준을 설명하세요. 전체 라벨 코드만으로 질문의 조건을 판단할 수 있나요?

**Q4.** 선택한 리뷰를 RAG에 활용한다면 어떤 순서로 처리하나요? 검색 결과를 LLM에 전달하는 목적은 무엇이며, 검색 결과와 최종 답변은 각각 무엇을 확인해야 하나요?

#### 제출 결과

- 점수·원본 연결 코드와 Top-1·Top-2 표
- Top-2의 실제 원문·기존 라벨·근거 출력
- 3가지 판단 기준을 적용한 `relevance_check` 표
- Q1~Q4 답변


### 풀이 전 안내 — 순위와 판단을 따로 기록하기

가상 점수는 문제에 주어진 값을 그대로 사용하세요. ID로 문서 정보를 연결한 다음 점수의 내림차순으로 정렬합니다. `head(K)`는 표의 앞 K행을 가져오므로 정렬 뒤에 사용해야 Top-K 결과가 됩니다.

원문을 읽을 때는 ‘패딩에 관한가’, ‘질문이 요구한 착용 상황을 직접 말하는가’, ‘해당 후기 사례를 찾는 데 충분한가’를 차례대로 확인하세요. 판단 표의 문구는 직접 작성하는 것이며 pandas가 자동으로 원문을 이해하는 것은 아닙니다.

답변은 ‘검색 순위와 점수 → 질문 조건 → 원문 근거 → 적합성 판단’ 순서로 쓰면 됩니다. 기존 전체 라벨 코드가 같아도 현재 질문에 대한 적합성은 다를 수 있습니다.


In [5]:
# 필수 2 코드를 작성하세요.
score_table = pd.DataFrame({
    'review_id': [1024338, 1024477, 1025044, 1025046],
    '가상_유사도': [0.81, 0.85, 0.91, 0.30]
})

search_results = score_table.merge(documents, on='review_id', how='left')

ranked_results = search_results.sort_values('가상_유사도', ascending=False)
top2 = ranked_results.head(2)
print(top2[['review_id','가상_유사도','product_name','source']].to_string(index=False))

for rid in top2['review_id']:
    show_review_context(rid)


 review_id  가상_유사도  product_name source
   1025044    0.91 OO 아** 구스코트      SNS
   1024477    0.85 OO 아** 구스코트      SNS

리뷰 ID: 1025044
메타데이터: {'source': 'SNS', 'domain': '패션', 'category': '여성의류', 'product_name': 'OO 아** 구스코트  '}
원문: 오늘도 정말 춥네요... 롱패딩 찾고 계신 분을 위한 후기 공유합니다.  키 158센티로에 48키로 마른 체형인데 90사이즈가 여유가 별로 없이 딱 맞습니다. 저는 딱 맞게 입기를 좋아해서 아주 만족해요. 와인색이 정말 특이하게 다운된 색인데 정말 고급스러워서 더욱 마음에 듭니다. 가격도 아주 저렴한 편이라 부담없이 사서 입게 되었답니다. 핏되게 나온 옷이라 넉넉하게 입기를 좋아하시는 분은 한 사이즈 올려 입으셔도 좋을 것 같아요. 저는 이 패딩으로 올겨울 아주 따뜻하게 잘 지내게 될 것 같습니다.   새로운 패딩 장만하고 올 겨울 따스하게 보내시길요...
기존 전체 라벨:
 review_id  sentiment_code
   1025044             1.0
속성별 주석:
 review_id  aspect_no aspect                                        evidence_text  sentiment_code
   1025044          1    사이즈     키 158센티로에 48키로 마른 체형인데 90사이즈가 여유가 별로 없이 딱 맞습니다.                1
   1025044          2    사이즈                             저는 딱 맞게 입기를 좋아해서 아주 만족해요               1
   1025044          3     색상            와인색이 정말 특이하게 다운된 색인데 정말 고급스러워서 더욱 마음에 듭니다    

### 필수 2 답변 작성란

**Q1.** Top-1과 Top-2의 리뷰 ID·점수는 무엇인가요? 0.91을 “정답일 확률 91%”로 해석할 수 있나요? 모든 검색 시스템에 같은 점수 기준을 적용해도 되나요?
|범위|순서|review_id|가상유사도|
|---|---|---|---|
|Top-1 및 Top-2|1|1025044|0.91|
|Top-2에 추가|2|1024477|0.85|
-> 0.91은 이 연습에서 제공한 가상 점수입니다. 실제 코사인 유사도라 하더라도 벡터 방향의 유사성을 나타내며 정답 확률 91%를 뜻하지 않는다.

**Q2.** Top-2 중 질문에 직접 적합한 리뷰는 무엇인가요? 두 리뷰의 실제 표현을 근거로 주제 관련성·조건 충족·정보 충분성을 설명하세요. “사이즈가 딱 맞는다”만으로 두꺼운 옷을 입기 불편하다고 확정할 수 있나요?
- 질문에 직접 적합한 문서는 1024477입니다.
- 1025044: 패딩과 사이즈를 다루므로 주제는 관련되어 있음. '90사이즈가 여유가 별로 없이 딱 맞습니다.'
- 이어서 '저는 딱 맞게 입기를 좋아해서 아주 만족한다.'는 설명이 있다.
- 두꺼운 옷을 안에 입었을 때의 불편을 직접 말하지 않아, 질문의 구체적인 조건을 뒷받침할 정보가 부족했다.

- 1024477: 안에 두껍게 입기에는 불편하다. 라는 직접적인 근거를 제시하고 있다.
- 이어서, 간편하게 니트 티셔츠에 입기에는 딱이라고 착용 조건을 구분하고 있다.
- 패딩이라는 주제, 두꺼운 안쪽 옷 착용 시 불편이라는 조건, 해당 후기 사례를 찾는 데 필요한 정보가 모두 확인된다.

> 사이즈가 딱 맞는다와 두꺼운 옷을 안에 입기 불편하다는 같은 진술의 내용이 아니다.

**Q3.** 이 가상 결과에서 K=1과 K=2의 차이는 무엇인가요? K를 늘리면 생기는 장단점과 K를 정할 기준을 설명하세요. 전체 라벨 코드만으로 질문의 조건을 판단할 수 있나요?
- K=1이면 1025044만 선택되어 직접적인 근거가 있는 1024477을 확인하지 못한다.
- k=2이면 필요한 문서가 후보에 포함된다. 다만 후보 두 개가 모두 적합한 것이라고 판단할 수는 없다.

**Q4.** 선택한 리뷰를 RAG에 활용한다면 어떤 순서로 처리하나요? 검색 결과를 LLM에 전달하는 목적은 무엇이며, 검색 결과와 최종 답변은 각각 무엇을 확인해야 하나요?
- 처리 순서는 사용자 질문 -> 관련 문서 검색 -> 검색 결과를 LLM에 전달 -> 답변 생성
- 위 사례에서는 1024477의 관련 내용을 답변의 근거로 활용할 수 있다. 
- 문서 ID, 원문, 메타 데이터의 연결을 유지하면 어떤 후기에서 가져온 근거인지 확인할 수 있다.
- 검색 단계에서는 주제, 조건, 정보의 충분성을 확인한다.
- 생성된 답변에서 핵심적인 내용인 '두껍게 입을 때 불편하다'는 원문을 정확히 반영했는지, 니트 티셔츠 착용과의 차이를 뒤집지 않았는 지, 모든 사람에게 불편했다고 해석하지는 않았는 지 확인한다.
- 적합한 문서와 답변을 찾았더라도 최종 답변이 반드시 정확한 것은 아니다.

---
## 과제 1. 새 질문의 결과 검증과 검색 실패 점검

### 문제 3-1. 두 조건을 함께 충족하는 후기 찾기

#### 문제 설명

필수 문제와 같은 방법으로 새 질문을 독립적으로 검증합니다. 후보 문서는 그대로 사용합니다.

> **흰색 패딩을 샀고, 입지 않은 듯 매우 가볍다고 평가한 후기를 찾아줘.**

먼저 후보 4개의 원문을 읽고, **흰색 구매와 매우 가벼운 착용감**이라는 두 조건을 모두 뒷받침하는 기대 문서 ID를 정하세요. 그 다음 아래 가상 결과와 비교합니다. 점수를 보고 기대 문서를 정하지 않습니다.

| review_id | 가상_유사도 |
|---:|---:|
| 1024338 | 0.90 |
| 1024477 | 0.65 |
| 1025044 | 0.40 |
| 1025046 | 0.82 |

이 점수도 실제 모델 결과가 아닌 학습용 가정입니다.

#### 요구사항

1. `show_review_context()`로 후보 4개를 확인하고, 새 질문의 기대 문서 ID를 `expected_review_id`에 저장하세요. 두 조건의 근거 표현도 적으세요.
2. 위 가상 점수로 `task_scores`를 만들고 `documents`와 ID로 연결하세요. 내림차순으로 정렬하여 `task_ranked`에 저장합니다.
3. Top-1과 Top-2의 ID·점수·상품명·출처를 출력하고, 각각 기대 문서가 포함되는지 `expected_review_id in 결과표["review_id"].tolist()`로 확인하세요.
4. Top-2 각각에 대해 `주제 관련성`, `조건 충족`, `정보 충분성`, `판단 근거`를 적은 `task_check` 표를 만드세요. 판단은 사람이 원문을 읽고 작성합니다.
5. 다음 두 행의 `validation_table`을 작성하세요. 각 질문의 기대 문서는 해당 질문에 직접적인 근거가 있는 문서로 정합니다.

| 질문 | 기대 문서 ID | 기대한 이유 | Top-1의 ID | Top-2의 ID 목록 | Top-1에 기대 문서 포함 | Top-2에 기대 문서 포함 |
|---|---|---|---|---|---|---|
| 필수 2 질문 | 직접 작성 | 원문 근거 | 검색 결과 | 검색 결과 | 확인 | 확인 |
| 과제 1 질문 | 직접 작성 | 원문 근거 | 검색 결과 | 검색 결과 | 확인 | 확인 |

#### 해석 질문

**Q1.** 과제의 기대 문서는 무엇이며 두 조건의 실제 근거는 무엇인가요? Top-1 문서가 질문에 충분히 적합하다고 보기 어려운 이유는 무엇인가요?

**Q2.** 두 질문에서 K=1과 K=2의 결과는 어떻게 달라지나요? 이 표만으로 실제 임베딩 모델의 성능이나 모든 질문에서의 검색 성공을 보장할 수 있나요?

**Q3.** 다음 상황마다 먼저 확인할 사항을 쓰세요. ① 필요한 문서가 없음 ② 관련 문서가 상위 K개에서 빠짐 ③ 관련 없는 문서가 반복 검색됨 ④ 한국어 질문으로 영어 문서 검색이 잘 안 됨 ⑤ 검색 결과와 현재 원문이 다름.

**Q4.** 사용자가 “이 상품의 공식 환불 기간은 며칠인가요?”라고 물었는데 이 후보 리뷰 4개만 검색되었다면, RAG에 전달할 정보가 충분한가요? 주제 관련성과 정보 충분성을 구분하고, 무엇을 추가로 확인해야 하는지 설명하세요.

#### 제출 결과

- 원문 확인, 기대 문서 선정, Top-K 선택 및 포함 여부 확인 코드
- `task_check`, `validation_table`
- Q1~Q4 답변


### 풀이 전 안내 — 점수보다 먼저 기준 정하기

새 질문은 두 조건을 동시에 요구합니다. 후보 원문을 먼저 읽어 각 조건의 근거를 찾고, 두 근거가 같은 리뷰에 있는지 확인한 뒤 기대 ID를 정하세요. 점수가 가장 높은 문서를 곧바로 기대 문서로 지정하지 않습니다.

`expected_review_id in 선택표["review_id"].tolist()`는 기대 ID가 선택된 ID 목록에 있는지 검사합니다. 이 표현은 의미를 판정하지 않으며 결과는 `True` 또는 `False`입니다. `True`일 때도 선택된 나머지 문서가 조건을 충족하는지 따로 설명하세요.

검증표에서는 한 행을 질문 하나로 구성합니다. 필수 2 결과와 새 질문의 결과를 각각 가져오고, 기대 문서·검색 순위·포함 여부를 섞지 않도록 변수 이름을 확인하세요. 새로운 모델이나 평가 지표를 구현하는 과제가 아닙니다.


In [ ]:
# 과제 1 코드를 작성하세요.


### 과제 1 답변 작성란

**Q1.**

- 답변:

**Q2.**

- 답변:

**Q3.**

- 답변:

**Q4.**

- 답변:


## 개념 정리 — 이번 강의에서 구분할 것

| 구분 | 첫 번째 의미 | 두 번째 의미 |
|---|---|---|
| 문서 ID / 저장 위치 | 원본 리뷰를 구별하는 값 | 벡터를 저장한다고 가정한 행 위치 |
| 본문 / 메타데이터 | 검색할 텍스트 내용 | 문서의 출처·분야·상품명 등 관련 정보 |
| 문서 임베딩 / 질문 임베딩 | 찾을 대상의 의미 표현 | 찾으려는 내용의 의미 표현 |
| 검색 점수 / 실제 적합성 | 후보의 순위를 정하는 수치 | 질문 조건을 원문이 뒷받침하는 정도 |
| Top-K / 조건에 맞는 문서 | 점수 기준으로 선택한 상위 K개 | 내용을 확인하여 활용할 수 있다고 판단한 문서 |
| 기대 문서 / Top-1 문서 | 원문을 기준으로 정한 비교 대상 | 검색 점수가 가장 높은 문서 |
| 기존 전체 라벨 / 검색 적합성 | 데이터셋에 있던 리뷰의 코드 | 현재 질문에 대한 별도의 판단 |
| 검색 검증 / 답변 검증 | 필요한 근거를 가진 문서인지 확인 | 생성된 답변이 근거를 정확히 반영하는지 확인 |

설명할 때는 ‘어떤 질문인지 → 어떤 결과가 선택되었는지 → 원문에 어떤 근거가 있는지 → 어느 범위까지 말할 수 있는지’의 순서로 작성해 보세요. 점수만 제시하거나 ‘관련 있다’고 끝내지 않고 질문의 구체적인 조건과 연결하는 것이 중요합니다.


---
## 실습 마무리

다음 질문에 자신의 말로 답해 보세요.

1. 이번 실습에서 해결하려던 검색 문제는 무엇이었나요?
2. 문서 본문·ID·메타데이터·가상 벡터 저장 위치는 어떻게 연결했나요?
3. Top-K 결과 중 활용할 문서를 무엇을 근거로 선택했나요?
4. 이번에 직접 실행한 작업과 가정한 검색 결과는 각각 무엇인가요?
5. 검색 결과를 RAG의 답변 근거로 활용하기 전에 무엇을 검증해야 하나요?


### 실습 마무리 답변 작성란

1. 이번 실습에서 해결하려던 검색 문제는 무엇이었나요?
- 패딩이라는 넓은 주제가 같은 리뷰들 중에서 특정 조건을 직접 말한 후기를 찾았다.
- 첫 질문은 두꺼운 옷을 안에 입을 때의 불편을
- 두 번째 질문은 흰색 구매와 매우 가벼운 착용감을 요구함
- 따라서 같은 후보 목록이라도 질문별로 적합한 문서가 달라짐

2. 문서 본문·ID·메타데이터·가상 벡터 저장 위치는 어떻게 연결했나요?
- CSV에서 원문, ID 관련 정보를 읽어 후보 네 개를 준비했다.
- 필수 1에서는 가상 저장 위치와 ID를 연결했고, 필수 2와 과제에서는 ID로 점수 표와 문서 표를 연결했다.
- 점수순으로 정렬해 Top-K를 고른 뒤 ID를 이용해 실제 원문과 주석을 조회했다.

3. Top-K 결과 중 활용할 문서를 무엇을 근거로 선택했나요?
- 첫 질문에서는 1위 1025044보다 2위 1024477에 직접적인 불편 표현이 있었다.
- 과제에서는 1위 1024338보다 2위 1025046에 흰색과 무게에 대한 두 조건의 근거가 있었다.
- 즉, 숫자는 확인할 순서를 알려 주었고, 최종 적합성은 질문과 원문을 비교하여 판단했다.

4. 이번에 직접 실행한 작업과 가정한 검색 결과는 각각 무엇인가요?
- 코드는 표를 읽고 연결하고 정렬하고 ID 포함 여부를 검사했습니다.
- 사람이 기대 문서를 정하고 근거 문구와 판단을 작성했다.
- relevance_check와 task_check는 사람이 적은 답안을 표로 보여준 것이지
- 코드가 원문의 의미를 이해해서 판정한 결과가 아니다.

5. 검색 결과를 RAG의 답변 근거로 활용하기 전에 무엇을 검증해야 하나요?
- 두 질문에서 Top-2에 기대 문서가 포함된다는 점은 확인했다.
- 하지만 후보 네 개와 가상 점수마으로 전체 리뷰 검색의 품질을 보장할 수는 없다.
- RAG에 활용할 때에도 질문에 필요한 근거를 확보했는지 확인하고
- 생성된 답변이 작성자의 경험을 확대하거나 조건을 뒤집지 않았는지 따로 살펴야한다.